In [ ]:
#Student_name = "Iman Dashtpeyma"😊
#Student_id = "imda3139"
#Student_background = "Software developer, Network security, and DevOps "

## Synthetic Dataset

### Step 1: Import Libraries and Define Functions

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN

# Generate the dataset
X, real_labels = make_moons(n_samples=100, noise=0.1, random_state=42)

# Standardize the dataset
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Function to calculate real centers
def calculate_real_cluster_centers(X, labels):
    df = pd.DataFrame(X, columns=['x1', 'x2'])
    df['label'] = labels
    return df.groupby('label')[['x1', 'x2']].mean().values

real_centers = calculate_real_cluster_centers(X_scaled, real_labels)

# Visualization function (your exact function)
def visualize_and_compare_clusters(X, real_labels, real_centers, estimated_labels, estimated_centers):
    """
    Input:
        X: (np.array[N,2]) - The dataset to visualize (only 2 features)
        real_labels: (np.array[N,1]) - Real cluster membership
        real_centers: (np.array[K,2]) - Real cluster centers
        estimated_labels: (np.array[N,1]) - Estimated cluster membership from algorithm
        estimated_centers: (np.array[K,2]) - Estimated cluster centers
    Returns:
        A plot with two axes showing the real clusters and the predicted clusters
    """
    fig, axes = plt.subplots(1, 2, figsize=(10,5))

    # First plot contains real cluster membership
    ax = axes[0]
    ax.scatter(x = X[:,0], y = X[:,1], c=real_labels, s=30, label= "Data")
    ax.scatter(x = real_centers[:,0], y = real_centers[:,1],
            c=np.arange(real_centers.shape[0]), s=200, marker="x", label="Centers")
    ax.set(xlabel="x1",ylabel="x2",title="Real clusters")
    ax.grid(True); ax.legend()

    # Second plot contains estimated cluster membership
    ax = axes[1]
    ax.scatter(x = X[:,0], y = X[:,1], c=estimated_labels, s=30, label="Data")
    ax.scatter(x = estimated_centers[:,0], y = estimated_centers[:,1],
            c=np.arange(estimated_centers.shape[0]), s=200, marker="x", label="Centers")
    ax.set(xlabel="x1",ylabel="x2",title="Estimated clusters")
    ax.grid(True); ax.legend()

    return plt.show()


##  1. K-Means (2 Clusters)

In [ ]:
# K-means with 2 clusters
kmeans_2 = KMeans(n_clusters=2, random_state=42)
estimated_labels_k2 = kmeans_2.fit_predict(X_scaled)
estimated_centers_k2 = kmeans_2.cluster_centers_

# Visualize
visualize_and_compare_clusters(X_scaled, real_labels, real_centers, estimated_labels_k2, estimated_centers_k2)


#### 1- Are the estimated clusters equal to the real clusters?
No, but they are quite close. There is a little bit of misclassification at the overlapping region between the two moons.
#### 2- Are the estimated centers close to the real centers?
Yes, they are relatively close, although slightly shifted due to the shape of the data not being perfectly spherical.
#### 3- Any problem caused by bad initialization?
 No obvious problems here because we set a random_state to ensure reproducibility and KMeans++ initializes smartly.
#### 4- Would the resulting clusters be useful, or do you need new hyperparameters or algorithms?
Useful but not perfect. Trying algorithms designed for non-linear shapes like DBSCAN or Spectral Clustering could be better.

## 2. K-Means (5 Clusters)

In [ ]:
# K-means with 5 clusters
kmeans_5 = KMeans(n_clusters=5, random_state=42)
estimated_labels_k5 = kmeans_5.fit_predict(X_scaled)
estimated_centers_k5 = kmeans_5.cluster_centers_

# Visualize
visualize_and_compare_clusters(X_scaled, real_labels, real_centers, estimated_labels_k5, estimated_centers_k5)


#### 1-Are the estimated clusters equal to the real clusters?
No. The clustering forces the data into 5 groups, which doesn't match the real moon shapes.
#### 2-Are the estimated centers close to the real centers?
No. The centers are spread out and do not reflect the underlying true structure.
#### 3-Any problem caused by bad initialization?
No bad initialization, but the wrong number of clusters (n_clusters=5) is the main issue.
#### 4-Would the resulting clusters be useful, or do you need new hyperparameters or algorithms?
No. 5 clusters don't make sense here. We must choose 2 clusters or switch to a clustering method like DBSCAN

## 3. DBSCAN

In [ ]:
dbscan = DBSCAN(eps=0.3, min_samples=5)
estimated_labels_dbscan = dbscan.fit_predict(X_scaled)

# Manually calculate DBSCAN centers (excluding noise points)
def calculate_dbscan_centers(X, labels):
    df = pd.DataFrame(X, columns=['x1', 'x2'])
    df['label'] = labels
    df = df[df['label'] != -1]  # Exclude noise points
    return df.groupby('label')[['x1', 'x2']].mean().values

estimated_centers_dbscan = calculate_dbscan_centers(X_scaled, estimated_labels_dbscan)

# Visualize
visualize_and_compare_clusters(X_scaled, real_labels, real_centers, estimated_labels_dbscan, estimated_centers_dbscan)


#### 1- Are the estimated clusters equal to the real clusters?
Very close! DBSCAN finds the two moon shapes naturally. A few points are marked as noise.
#### 2-Are the estimated centers close to the real centers?
Yes, the centers are closer compared to K-means results.
#### 3-Any problem caused by bad initialization?
No. DBSCAN is deterministic given the parameters eps and min_samples.
#### 4-Would the resulting clusters be useful, or do you need new hyperparameters or algorithms?
Mostly useful! Maybe fine-tuning eps could slightly reduce the noise points. Otherwise, it’s a good fit.

### KMeans (2 clusters) = okay but struggles with non-linear data.
### KMeans (5 clusters) = wrong hyperparameter, bad clustering.
### DBSCAN = best fit for this type of moon-shaped data.

## Clustering your real-life dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Load the dataset
data = pd.read_csv('./data/combined_student_data.csv')

# Select two numerical features for clustering
features = ["G1", "G2"]
X = data[features]

# Preprocessing: Drop missing values and ensure correct types
X = X.dropna().astype(float)

# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Experimental evaluation (range: 2 to 10 clusters)
silhouette_scores = []
range_n_clusters = range(2, 11)

for n_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(score)

# Plot silhouette scores vs number of clusters
plt.figure(figsize=(8, 6))
plt.plot(range_n_clusters, silhouette_scores, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs Number of Clusters')
plt.grid(True)
plt.show()

# Determine the best number of clusters
best_k = range_n_clusters[np.argmax(silhouette_scores)]
print(f"Best number of clusters: {best_k}")

# Clustering with the optimal number of clusters
optimal_kmeans = KMeans(n_clusters=best_k, random_state=42)
final_clusters = optimal_kmeans.fit_predict(X_scaled)

# Visualization of final clusters
plt.figure(figsize=(8, 6))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=final_clusters, cmap='viridis', edgecolor='k', s=60)
plt.xlabel(features[0])
plt.ylabel(features[1])
plt.title(f'Final KMeans Clustering (k={best_k})')
plt.grid(True)
plt.show()


### 🤔 1. Which number of clusters is best to group the samples based on your features?
The best number of clusters is 2, as indicated by the highest silhouette score. This number optimally balances cluster separation and cohesion, clearly delineating groups within the chosen features "G1" and "G2".

### 🤔 2. How well did the algorithm cluster the groups based on the selected 2 features?
The clustering algorithm effectively separated the data into two distinct groups, reflected by the highest silhouette score achieved at k=2. This suggests good internal consistency within each cluster and clear differentiation between them. However, further analysis or including additional features could potentially reveal more nuanced groupings.

### 🤔 3. Would you deploy this clustering model to a client as a final solution to group observations in your dataset?
While the current model provides a clear initial segmentation based on two academic scores ("G1", "G2"), deploying this as a final solution may be premature. Additional validation, potentially with other features or different clustering algorithms, would be beneficial to ensure robustness and applicability.